In [ ]:
# If you're in Colab, run this cell first
!pip -q install tabpfn openml scikit-learn pandas numpy

import torch, sys
print("PyTorch CUDA available:", torch.cuda.is_available(), "| Python", sys.version)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import torch

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)

clf = TabPFNClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
clf.fit(Xtr, ytr)

proba = clf.predict_proba(Xte)[:, 1]
pred  = (proba >= 0.5).astype(int)
print("Accuracy:", accuracy_score(yte, pred))
print("ROC AUC:", roc_auc_score(yte, pred))

## details of breast cancer dataset here
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic 

## Let's look at the UCI Parkinson’s “voice” dataset. It’s tiny (195 recordings, ~22 numeric features) and the label is just “status” (0 = healthy, 1 = PD). Remember that rows are independent voice recordings, not time samples

## 

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

ds = fetch_openml("parkinsons", version=1, as_frame=True)
X = ds.data.drop(columns=["name"], errors="ignore")
y = ds.target.astype(int) - 1

X.shape, y.value_counts().to_dict()

## Let's first do some data exploration- let's visualize the data and get an idea of what's going on

In [ ]:
# Code
# Here
# !

In [ ]:
import numpy as np, pandas as pd

df = X.copy()
df["status"] = y.values
print("Rows:", len(df), "| Features:", X.shape[1])
print("Class balance:", df["status"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
import numpy as np, pandas as pd

y_centered = (y - y.mean()).values
corrs = {}
for c in X.columns:
    xc = X[c].values.astype(float)
    if np.std(xc) == 0: 
        corrs[c] = 0.0
    else:
        corrs[c] = float(np.corrcoef(xc, y_centered)[0,1])
corr = pd.Series(corrs).abs().sort_values(ascending=False)
corr.head(10)


In [ ]:
import matplotlib.pyplot as plt

top_feats = list(corr.head(3).index)
for feat in top_feats:
    plt.figure(figsize=(5,4))
    X.loc[y==0, feat].plot(kind="hist", bins=30, alpha=0.5, density=True, label="status=0")
    X.loc[y==1, feat].plot(kind="hist", bins=30, alpha=0.5, density=True, label="status=1")
    plt.title(f"Distribution by class: {feat}")
    plt.xlabel(feat); plt.ylabel("density"); plt.legend()
    plt.tight_layout()


In [ ]:
import numpy as np, matplotlib.pyplot as plt

C = np.corrcoef(X.values.astype(float), rowvar=False)
plt.figure(figsize=(6,5))
im = plt.imshow(C, aspect="auto")
plt.title("Feature–feature correlation")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(X.columns)), X.columns, rotation=90, fontsize=7)
plt.yticks(range(len(X.columns)), X.columns, fontsize=7)
plt.tight_layout()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

sc = StandardScaler().fit(X)
Z = sc.transform(X)
pca = PCA(n_components=2, random_state=0).fit(Z)
Z2 = pca.transform(Z)

plt.figure(figsize=(5,4))
m0 = y.values==0; m1 = ~m0
plt.scatter(Z2[m0,0], Z2[m0,1], s=18, marker="o", alpha=0.6, label="status=0")
plt.scatter(Z2[m1,0], Z2[m1,1], s=18, marker="x", alpha=0.7, label="status=1")
evr = pca.explained_variance_ratio_.sum()
plt.title(f"PCA (2D) — EVR={evr:.2f}")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.tight_layout()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

## Let's fit a simple classifier first- logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=2000).fit(X_tr_s, y_tr)
proba_lr = lr.predict_proba(X_te_s)[:, 1]
print("LogReg  | Acc:", accuracy_score(y_te, proba_lr>=0.5), "| ROC AUC:", roc_auc_score(y_te, proba_lr))

In [ ]:
from tabpfn import TabPFNClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
clf = TabPFNClassifier(device=device)  # drop-in sklearn-style API
clf.fit(X_tr_s, y_tr)

proba = clf.predict_proba(X_te_s)[:, 1]
pred  = (proba >= 0.5).astype(int)

print("TabPFN  | Acc:", accuracy_score(y_te, pred), "| ROC AUC:", roc_auc_score(y_te, proba))

## Let's look at cross validation

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def tabpfn_cv_auc(X_df, y_ser, n_splits=5, seed=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aucs = []
    for tr, te in skf.split(X_df, y_ser):
        X_tr, X_te = X_df.iloc[tr], X_df.iloc[te]
        y_tr, y_te = y_ser.iloc[tr], y_ser.iloc[te]
        scaler = StandardScaler().fit(X_tr)
        X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)
        model = TabPFNClassifier(device=device)
        model.fit(X_tr_s, y_tr)
        proba = model.predict_proba(X_te_s)[:, 1]
        aucs.append(roc_auc_score(y_te, proba))
    return np.mean(aucs), np.std(aucs)

mean_auc, std_auc = tabpfn_cv_auc(X, y, n_splits=5, seed=42)
print(f"TabPFN CV ROC AUC: {mean_auc:.3f} ± {std_auc:.3f}")


# Go to their tutorial for more! https://colab.research.google.com/github/PriorLabs/TabPFN/blob/main/examples/notebooks/TabPFN_Demo_Local.ipynb